In [ ]:
import sys

sys.path.insert(
    0, "."
)  # lets the notebook import base.py when started from the repo root or impl/

import base
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Audio

## Case 1: Cat & Dog sound classification (machine learning)
Dataset: 277 short clips (Google Drive, ~100 MB), downloaded into `data/project_1/case_1`.

In [ ]:
from sklearn.feature_selection import f_classif

In [ ]:
c1, SR = base.Case(1, 1), 22050
base.download_gdrive_zip("16T1uGEtgFPHMCxl-H1NgaemjtxAVpUoZ", c1.data)

root = c1.data / "cats_dogs"
folders = {
    ("train", "cat"): "train/cat",
    ("train", "dog"): "train/dog",
    ("test", "cat"): "test/cats",
    ("test", "dog"): "test/test",
}

df = pd.DataFrame(
    [
        (s, l, p)
        for (s, l), f in folders.items()
        for p in sorted((root / f).glob("*.wav"))
    ],
    columns=["split", "label", "path"],
)

df.groupby(["split", "label"]).size().unstack()

In [ ]:
# EDA: waveform + mel spectrogram of one clip per class
fig, ax = plt.subplots(2, 2, figsize=(11, 5))
for j, lab in enumerate(["cat", "dog"]):
    y = base.load_audio(df[(df.label == lab) & (df.split == "train")].path.iloc[0], SR)
    librosa.display.waveshow(y, sr=SR, ax=ax[0, j])
    ax[0, j].set_title(f"{lab}: waveform")
    S = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR), ref=np.max)
    librosa.display.specshow(S, sr=SR, x_axis="time", y_axis="mel", ax=ax[1, j])
    ax[1, j].set_title(f"{lab}: mel spectrogram")
    
c1.save_fig("eda")

In [ ]:
# Features (cached): MFCC/spectral/pitch/wavelet statistics per clip
cache = c1.data / "features.csv"
if cache.exists():
    feats = pd.read_csv(cache)
else:
    feats = base.build_features(df.path, SR)
    feats.to_csv(cache, index=False)

X = feats.drop(columns="source")
y = (df.label == "dog").astype(int).values
tr, te = (df.split == "train").values, (df.split == "test").values
X_train, X_test, y_train, y_test = X[tr], X[te], y[tr], y[te]

# Most discriminative features per class (ANOVA F-score)
top = X_train.columns[np.argsort(-np.nan_to_num(f_classif(X_train, y_train)[0]))[:8]]
long = X_train[top].assign(label=np.where(y_train == 1, "dog", "cat")).melt("label")
sns.catplot(
    long,
    x="label",
    y="value",
    col="variable",
    col_wrap=4,
    kind="box",
    sharey=False,
    height=2.4,
)

c1.save_fig("top_features")

Models are tuned with 5-fold CV on the training clips only; the test clips are used once for the final score.
The best model (highest CV macro-F1, faster predictor on ties) is saved in `model/`.

In [ ]:
res, best = base.compare_models(c1, X_train, y_train, X_test, y_test, ["cat", "dog"])
res.round(3)

Inference example with the saved model

In [ ]:
sample = df[df.split == "test"].sample(2, random_state=base.SEED)
for p, true in zip(sample.path, sample.label):
    print(f"{p.name} (true: {true}) ->", base.predict_clip(c1, p, SR))
    
Audio(base.load_audio(sample.path.iloc[0], SR), rate=SR)

## Case 2: Cough audio classification (deep learning)
Dataset (Google Drive zip, **~1.3 GB**, downloaded once): healthy / symptomatic / COVID-19 coughs.
Only the selected clips are decoded; log-mel spectrograms are cached and the zip is deleted afterwards.
Note: the labels are self-reported, so accuracy well above chance (33%) is hard to reach.

In [ ]:
import json
import time
import zipfile
from pathlib import Path

import gdown
import soundfile as sf
import torch
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from tqdm.auto import tqdm

In [ ]:
c2 = base.Case(1, 2)
SR2, SECONDS, N_MELS, HOP = 16000, 8, 64, 256
N_PER_CLASS = 300  # None = all (1155 per class); set e.g. 300 for a quick run

torch.manual_seed(base.SEED)
np.random.seed(base.SEED)
DEVICE = (
    "cuda" if torch.cuda.is_available() else "cpu"
)  # runs on CPU as well, just slower
print("device:", DEVICE)


def logmel(y):
    """Fixed-length (8 s) log-mel spectrogram, shape (N_MELS, frames)."""
    y = np.pad(y, (0, max(0, SR2 * SECONDS - len(y))))[: SR2 * SECONDS]
    return librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=y, sr=SR2, n_fft=1024, hop_length=HOP, n_mels=N_MELS
        )
    ).astype("float32")


cache = c2.data / "logmel.npz"
if not cache.exists():
    zpath = c2.data / "cough.zip"
    if not zpath.exists():
        gdown.download(
            id="1acT-syyiK6PI7h43i7-o8rgukYTf_jQj", output=str(zpath), quiet=False
        )
    with zipfile.ZipFile(zpath) as z:
        members = {
            Path(n).stem: n
            for n in z.namelist()
            if n.lower().endswith((".ogg", ".webm"))
        }

        meta = pd.read_csv(
            z.open(
                next(n for n in z.namelist() if n.endswith("metadata_compiled.csv"))
            ),
            usecols=["uuid", "status"],
        )

        meta = meta.dropna(subset=["status"])
        meta = meta[meta.uuid.isin(members)]

        n = (
            N_PER_CLASS or meta.status.value_counts().min()
        )  # undersample to a balanced set

        meta = (
            meta.groupby("status")
            .sample(n=min(n, meta.status.value_counts().min()), random_state=base.SEED)
            .reset_index(drop=True)
        )

        (c2.data / "examples").mkdir(
            exist_ok=True
        )  # one real clip per class for the inference demo

        X, seen = [], set()
        for u, s in tqdm(zip(meta.uuid, meta.status), total=len(meta), desc="log-mel"):
            w = base.load_audio(z.read(members[u]), SR2)
            X.append(logmel(w))
            if s not in seen:
                seen.add(s)
                sf.write(c2.data / "examples" / f"{s}.wav", w, SR2)

    np.savez_compressed(
        cache, X=np.stack(X), status=meta.status.values, uuid=meta.uuid.values
    )
    zpath.unlink()  # the 1.3 GB zip is no longer needed

d = np.load(cache, allow_pickle=True)
X, classes = d["X"], sorted(set(d["status"]))
y = np.array([classes.index(s) for s in d["status"]])
print(X.shape, dict(zip(classes, np.bincount(y))))

In [ ]:
# Stratified 80/10/10 split; normalise with train statistics only
idx = np.arange(len(y))
i_tr, i_tmp = train_test_split(idx, test_size=0.2, stratify=y, random_state=base.SEED)
i_va, i_te = train_test_split(
    i_tmp, test_size=0.5, stratify=y[i_tmp], random_state=base.SEED
)

mean, std = float(X[i_tr].mean()), float(X[i_tr].std())
tensor = lambda ix: torch.tensor((X[ix] - mean) / std).unsqueeze(1)
data = {
    k: (tensor(ix), torch.tensor(y[ix]))
    for k, ix in {"train": i_tr, "val": i_va, "test": i_te}.items()
}

loader = lambda k, bs=32: torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(*data[k]), batch_size=bs, shuffle=(k == "train")
)

fig, ax = plt.subplots(1, 3, figsize=(12, 2.8))
for a, c in zip(ax, range(3)):
    a.imshow(X[y == c][0], origin="lower", aspect="auto", cmap="magma")
    a.set_title(f"{classes[c]} (log-mel)")

c2.save_fig("samples")

Compact CNN (4 conv blocks + global average pooling, ~60k params)
with SpecAugment-style masking, AdamW, LR scheduling and early stopping. A small grid picks width and learning rate on the validation set.

In [ ]:
class CoughCNN(nn.Module):
    def __init__(self, n_classes=3, width=16, dropout=0.3):
        super().__init__()
        block = lambda i, o: nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.features = nn.Sequential(
            block(1, width),
            block(width, 2 * width),
            block(2 * width, 4 * width),
            block(4 * width, 4 * width),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * width, n_classes))

    def forward(self, x):
        if (
            self.training
        ):  # random frequency / time masks (value 0 = mean after normalisation)
            f, t = np.random.randint(0, 9), np.random.randint(0, 41)
            f0, t0 = (
                np.random.randint(0, N_MELS - f + 1),
                np.random.randint(0, x.shape[3] - t + 1),
            )
            x = x.clone()
            x[:, :, f0 : f0 + f], x[:, :, :, t0 : t0 + t] = 0, 0
        return self.head(self.features(x).mean((2, 3)))


runs = []
for width in (16, 32):
    for lr in (1e-3, 3e-4):
        print(f"\n== width={width} lr={lr}")
        t0 = time.perf_counter()
        m, hist = base.train_classifier(
            CoughCNN(width=width),
            loader("train"),
            loader("val"),
            epochs=20,
            lr=lr,
            patience=5,
            device=DEVICE,
        )

        runs.append(
            {
                "width": width,
                "lr": lr,
                "val_loss": hist.val_loss.min(),
                "val_acc": hist.val_acc[hist.val_loss.idxmin()],
                "train_s": time.perf_counter() - t0,
                "model": m,
                "hist": hist,
            }
        )

summary = pd.DataFrame(runs).drop(columns=["model", "hist"])
best_run = runs[int(summary.sort_values(["val_loss", "train_s"]).index[0])]
summary.round(3)

In [ ]:
base.plot_history(best_run["hist"], c2)
model = best_run["model"]
p = base.predict_torch(model, data["test"][0])
yt = data["test"][1].numpy()
plt.figure(figsize=(4, 3.5))
base.plot_confusion(
    yt, p.argmax(1), classes, f"CoughCNN test (acc {(p.argmax(1) == yt).mean():.2%})"
)

c2.save_fig("confusion_test")
print(classification_report(yt, p.argmax(1), target_names=classes))

# Save the best model + everything needed for inference
torch.save(model.cpu().state_dict(), c2.model / "model.pt")
meta = {
    "width": best_run["width"],
    "classes": classes,
    "mean": mean,
    "std": std,
    "sr": SR2,
    "seconds": SECONDS,
    "n_mels": N_MELS,
    "hop": HOP,
    "val_acc": best_run["val_acc"],
    "test_acc": float((p.argmax(1) == yt).mean()),
    "train_seconds": best_run["train_s"],
}

c2.save_json("meta", meta, model=True)

Inference example with the saved model

In [ ]:
def predict_cough(wave):
    """wave: 16 kHz mono waveform -> {class: probability}"""
    meta = json.loads((c2.model / "meta.json").read_text())
    net = CoughCNN(len(meta["classes"]), meta["width"])
    net.load_state_dict(torch.load(c2.model / "model.pt"))
    x = torch.tensor((logmel(wave) - meta["mean"]) / meta["std"])[None, None]
    return dict(zip(meta["classes"], base.predict_torch(net, x)[0].round(3).tolist()))


for clip in sorted((c2.data / "examples").glob("*.wav")):
    print(f"{clip.stem:12s} ->", predict_cough(base.load_audio(clip, SR2)))